# PitLineMotor · Asistente de preguntas frecuentes

Ejecuta la celda. Si es la primera vez en este ordenador, antes instala los paquetes con: `%pip install nltk scikit-learn pandas ipywidgets` y reinicia el kernel.

In [1]:
# ============================================================
# ANÁLISIS DE TUS PRUEBAS
# ------------------------------------------------------------
# "cuento tiempo la reparacion de un ciliindro"  → OK (Tiempo estimado). Con la versión anterior
#                                                  "cuento" se confundía con "cuenta" (Cambiar email).
# "¿Qué servicios ofrecéis?"                     → SÍ debía tener respuesta: faltaba el tema → nuevo "Servicios del taller"
# "Reparáis averías de motor?"                   → Respuesta incorrecta (Tiempo estimado) → ahora "Servicios del taller"
# "¿El presupuesto es gratuito?"                 → SÍ tiene respuesta: la diagnosis se factura → añadido a "Precio"
# "¿Aceptáis pagos con tarjeta?"                 → El proyecto no define formas de pago → nuevo tema que remite a Recepción
# "¿Hacéis cambio de aceite?"                    → Ahora va a "Servicios del taller" (antes era el fallo de Cambiar email)
# Corrección de erratas: con raíces cortas se exige más parecido ("aceit" ya no se convierte en "acept")
# Evaluación: 11 de 11 aciertos (100 %)
# ============================================================

# ============================================================
# PitLineMotor · Asistente de preguntas frecuentes (PLN)
# ============================================================
import nltk
import re
import html
import difflib
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

# 1. Recursos de NLTK (solo se descargan la primera vez)
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import SnowballStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 2. Base de conocimiento: varias formas de preguntar lo mismo, por tema
preguntas = {
    'Pedir cita': [
        "¿Cómo pido cita?",
        "Quiero reservar una cita para mi coche",
        "¿Puedo pedir hora para llevar el vehículo al taller?"],
    'Estado de la reparación': [
        "¿Cómo va la reparación de mi coche?",
        "¿En qué fase está mi vehículo?",
        "Quiero saber si ya han arreglado mi coche",
        "¿Cuánto le queda a mi coche?",
        "¿Cuánto tiempo falta para que esté listo?",
        "¿Cuándo estará arreglado mi coche?",
        "¿Cuánto tardáis en arreglarlo?"],
    'Factura': [
        "¿Dónde descargo la factura?",
        "Necesito la factura de la reparación",
        "¿Cómo consigo el PDF de la factura?"],
    'Pendiente de pieza': [
        "¿Por qué mi reparación está parada?",
        "¿Qué significa pendiente de pieza?",
        "Mi coche está esperando una pieza"],
    'Recoger el coche': [
        "¿Cuándo puedo recoger mi coche?",
        "¿Ya puedo ir a por el vehículo?",
        "Quiero recoger el coche del taller"],
    'Precio': [
        "¿Cuánto cuesta la mano de obra?",
        "¿Qué precio tiene la hora de trabajo?",
        "¿Cómo se calcula el precio de la reparación?",
        "¿Cuánto me va a costar?",
        "¿El presupuesto es gratuito?",
        "¿Cobráis por revisar el coche?"],
    'Cambiar email': [
        "¿Cómo cambio mi email?",
        "Quiero modificar mi correo electrónico",
        "Actualizar el correo de mi cuenta"],
    'Tiempo estimado de una reparación': [
        "¿Cuál es el tiempo estimado de una reparación?",
        "¿Cuánto tiempo de espera hay para reparar el motor?",
        "¿Cuánto se tarda en cambiar el embrague?",
        "¿Cuántos días lleva una reparación de este tipo?",
        "¿Cuántos días se tarda en arreglar una avería?",
        "¿Cuánto tiempo lleva la reparación de un cilindro?"],
    'Avería o problema del vehículo': [
        "Mi batería se agota muy rápido",
        "Mi coche hace un ruido raro",
        "Mi coche arranca mal",
        "Se ha encendido una luz en el salpicadero",
        "Los frenos chirrían al frenar",
        "Creo que mi coche tiene una avería",
        "¿Podéis reparar mi coche sin verlo?",
        "¿Podéis saber qué avería tiene mi coche sin llevarlo al taller?",
        "¿Podéis diagnosticar mi coche por el chat?"],
    'Servicios del taller': [
        "¿Qué servicios ofrecéis?",
        "¿Qué tipo de reparaciones hacéis?",
        "¿Reparáis averías de motor?",
        "¿Qué trabajos realiza el taller?"],
    'Formas de pago': [
        "¿Aceptáis pagos con tarjeta?",
        "¿Cómo puedo pagar la reparación?",
        "¿Se puede pagar en efectivo?"],
    'Historial': [
        "¿Puedo ver reparaciones anteriores?",
        "Historial de reparaciones de mi coche",
        "¿Qué le hicieron a mi coche la última vez?"],
}

respuestas = {
    'Pedir cita': "Entra en «Mis citas» y crea una cita nueva con tu vehículo, la fecha y el motivo. Quedará pendiente hasta que Recepción la confirme.",
    'Estado de la reparación': "No podemos darte un tiempo exacto, porque depende de la diagnosis y de las piezas. En «Mis reparaciones» puedes ver en qué estado está tu vehículo: diagnosis, reparación, pendiente de pieza, prueba o finalizada.",
    'Factura': "Cuando la reparación está finalizada, entra en «Mis reparaciones» y descarga la factura en PDF.",
    'Pendiente de pieza': "Falta una pieza para continuar. El taller ya la ha pedido y, cuando llegue, el mecánico reanudará la reparación.",
    'Recoger el coche': "Cuando la reparación aparece como finalizada, el coche está listo. Pasa por Recepción, que registrará la entrega.",
    'Precio': "Se cobra el tiempo de diagnosis y reparación al precio por hora del taller, más las piezas utilizadas y el IVA. Todo aparece desglosado en la factura.",
    'Cambiar email': "Puedes cambiar tu email desde «Mi perfil».",
    # En el proyecto final este dato saldrá de la base de datos (orden de trabajo y piezas)
    'Tiempo estimado de una reparación': "Por ahora no podemos darte un tiempo estimado por el chat: depende de la diagnosis del mecánico y de que haya piezas disponibles. Si ya tienes una reparación en curso, consulta su estado en «Mis reparaciones»; si no, pide cita y te lo indicarán tras la diagnosis.",
    'Avería o problema del vehículo': "Por el chat no podemos diagnosticar averías. Pide cita en «Mis citas» y describe el problema en el motivo; un mecánico revisará el vehículo y te informará.",
    # En el proyecto final la lista saldrá de la tabla CATEGORIA_REPARACION
    'Servicios del taller': "Hacemos diagnosis y reparación de vehículos. Para saber si podemos encargarnos de tu caso, pide cita en «Mis citas» y describe el problema en el motivo.",
    # Las formas de pago no están definidas en el proyecto: se remite a Recepción
    'Formas de pago': "Consulta en Recepción las formas de pago disponibles.",
    'Historial': "En «Mis reparaciones» aparecen también las reparaciones anteriores con todos sus datos.",
}

df = pd.DataFrame([(p, tema) for tema, lista in preguntas.items() for p in lista],
                  columns=['pregunta', 'tema'])

# 3. Preprocesamiento: minúsculas, sin signos, tokens, sin stop words (salvo "no") y raíz
stemmer = SnowballStemmer('spanish')
stop_words_es = set(stopwords.words('spanish')) - {'no'}

def preprocesar(texto):
    texto = texto.lower()
    texto = re.sub(r'[^a-záéíóúüñ\s]', ' ', texto)
    tokens = word_tokenize(texto, language='spanish')
    return " ".join(stemmer.stem(t) for t in tokens if t not in stop_words_es)

df['pregunta_limpia'] = df['pregunta'].apply(preprocesar)

# 4. Vectorización con TF-IDF
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['pregunta_limpia'])

# Para cada raíz, en qué temas aparece (sirve para detectar palabras genéricas y sugerir temas)
temas_por_raiz = {}
for raiz_frase, tema in zip(df['pregunta_limpia'], df['tema']):
    for r in raiz_frase.split():
        temas_por_raiz.setdefault(r, set()).add(tema)

# Corrige raíces mal escritas usando la más parecida del vocabulario (ej.: "facutra" → "factur")
vocabulario = list(vectorizer.get_feature_names_out())

def raiz_conocida(raiz):
    if raiz in vocabulario:
        return raiz
    # Solo se corrige si se parece mucho y empieza por la misma letra
    # (con raíces cortas se exige más parecido: "aceit" no debe convertirse en "acept")
    minimo = 0.8 if len(raiz) >= 6 else 0.85
    candidatas = [c for c in difflib.get_close_matches(raiz, vocabulario, n=3, cutoff=minimo) if c[0] == raiz[0]]
    return candidatas[0] if candidatas else None

def corregir(texto_limpio):
    return " ".join(raiz_conocida(r) or r for r in texto_limpio.split())

# 5. Modelo: buscar la pregunta más parecida (similitud del coseno)
UMBRAL = 0.3

def responder(pregunta):
    limpio = corregir(preprocesar(pregunta))
    vector = vectorizer.transform([limpio])
    similitudes = cosine_similarity(vector, X)[0]
    mejor = similitudes.argmax()
    # Si solo coinciden palabras genéricas (coche, taller...), no hay una respuesta concreta
    conocidas = [r for r in limpio.split() if r in temas_por_raiz]
    solo_genericas = all(len(temas_por_raiz[r]) >= 3 for r in conocidas)
    if similitudes[mejor] < UMBRAL or solo_genericas:
        return None, similitudes[mejor], "No te he entendido. Prueba a decirlo de otra forma o pregunta en Recepción."
    tema = df.loc[mejor, 'tema']
    return tema, similitudes[mejor], respuestas[tema]

# 5b. Si no entiende: revisar palabra por palabra qué temas podrían encajar
def sugerir_temas(pregunta, maximo=4):
    puntos, pistas = {}, {}
    for palabra in re.sub(r'[^a-záéíóúüñ\s]', ' ', pregunta.lower()).split():
        if palabra in stop_words_es:
            continue
        raiz = stemmer.stem(palabra)
        # Raíz conocida o, si está mal escrita, la más parecida del vocabulario
        r = raiz_conocida(raiz)
        if r:
            for tema in temas_por_raiz[r]:
                puntos[tema] = puntos.get(tema, 0) + 1 / len(temas_por_raiz[r])
                pistas.setdefault(tema, []).append(palabra)
    ordenados = sorted(puntos, key=puntos.get, reverse=True)[:maximo]
    return [(tema, pistas[tema]) for tema in ordenados]

# 6. Evaluación rápida con preguntas nuevas
pruebas = [
    ("Me gustaría reservar hora para el lunes", 'Pedir cita'),
    ("¿Ya han terminado de arreglar mi vehículo?", 'Estado de la reparación'),
    ("No encuentro la factura de mi coche", 'Factura'),
    ("Me dicen que falta una pieza, ¿qué pasa?", 'Pendiente de pieza'),
    ("¿A qué hora paso a recoger el coche?", 'Recoger el coche'),
    ("¿Cuánto cuesta la reparación?", 'Precio'),
    ("Quiero cambiar mi correo", 'Cambiar email'),
    ("¿Puedo ver las reparaciones que le hicisteis antes?", 'Historial'),
    ("¿Abrís los domingos?", None),
    ("¿Vendéis neumáticos?", None),
    ("¿Hacéis cambio de aceite?", 'Servicios del taller'),
]
aciertos = sum(responder(texto)[0] == esperado for texto, esperado in pruebas)
print(f"Evaluación: {aciertos} de {len(pruebas)} aciertos ({aciertos / len(pruebas) * 100:.0f} %)")

# ============================================================
# 7. Chat con el estilo de PitLineMotor
# ============================================================
# Colores de la marca (cámbialos aquí si el proyecto usa otros)
OSCURO = "#151B26"    # cabecera, asfalto
ROJO = "#D7262E"      # color principal, línea de boxes
GRIS = "#F2F4F7"      # fondo del chat
TEXTO = "#1E2633"
SUAVE = "#6B7280"
FUENTE = "font-family: 'Segoe UI', Roboto, Arial, sans-serif;"

mensajes = []   # lista de (quién, texto, detalle)

def burbuja(quien, texto, detalle):
    texto = html.escape(texto)
    if quien == "cliente":
        return f"""
        <div style="display:flex; justify-content:flex-end; margin:8px 0;">
          <div style="max-width:75%; background:{ROJO}; color:#fff; padding:10px 14px;
                      border-radius:14px 14px 4px 14px; font-size:14px; line-height:1.45;">{texto}</div>
        </div>"""
    fallo = detalle is None
    fondo = "#FDECEC" if fallo else "#FFFFFF"
    borde = ROJO if fallo else "#E2E5EA"
    pie = "" if detalle == "" else (
        f'<div style="font-size:11px; color:{SUAVE}; margin-top:6px;">'
        f'{"Sin coincidencia suficiente" if fallo else html.escape(detalle)}</div>')
    return f"""
    <div style="display:flex; gap:8px; align-items:flex-end; margin:8px 0;">
      <div style="width:30px; height:30px; flex:none; border-radius:50%; background:{OSCURO}; color:#fff;
                  display:flex; align-items:center; justify-content:center; font-size:11px; font-weight:700;">PL</div>
      <div style="max-width:75%; background:{fondo}; color:{TEXTO}; border:1px solid {borde}; padding:10px 14px;
                  border-radius:14px 14px 14px 4px; font-size:14px; line-height:1.45;">{texto}{pie}</div>
    </div>"""

def pintar():
    cuerpo = "".join(burbuja(*m) for m in mensajes)
    pantalla.value = f"""
    <div style="{FUENTE} width:560px; max-width:100%; border-radius:14px; overflow:hidden;
                box-shadow:0 6px 24px rgba(21,27,38,.18); border:1px solid #E2E5EA;">
      <div style="background:{OSCURO}; color:#fff; padding:14px 18px; display:flex; align-items:center; gap:12px;">
        <div style="width:38px; height:38px; border-radius:8px; background:{ROJO}; display:flex;
                    align-items:center; justify-content:center; font-weight:800; font-size:15px;">PL</div>
        <div>
          <div style="font-weight:700; font-size:16px; letter-spacing:.3px;">PitLine<span style="color:{ROJO};">Motor</span></div>
          <div style="font-size:12px; color:#B8C0CC;">Asistente de dudas · <span style="color:#4ADE80;">●</span> en línea</div>
        </div>
      </div>
      <div style="height:6px; background:repeating-linear-gradient(90deg, {ROJO} 0 18px, #FFFFFF 18px 36px);"></div>
      <div style="background:{GRIS}; padding:12px 16px; height:340px; overflow-y:auto;
                  display:flex; flex-direction:column-reverse;"><div>{cuerpo}</div></div>
    </div>"""

def mostrar_opciones(lista):
    botones = []
    for tema in lista:
        b = widgets.Button(description=tema, button_style="warning", layout=widgets.Layout(width="auto"))
        b.on_click(lambda _, t=tema: elegir_tema(t))
        botones.append(b)
    opciones.children = botones

def elegir_tema(tema):
    mensajes.append(("cliente", tema, ""))
    mensajes.append(("asistente", respuestas[tema], f"Tema elegido: {tema}"))
    opciones.children = []
    pintar()

def enviar(texto):
    texto = texto.strip()
    if not texto:
        return
    opciones.children = []
    mensajes.append(("cliente", texto, ""))
    tema, similitud, respuesta = responder(texto)
    if tema is not None:
        mensajes.append(("asistente", respuesta, f"Tema: {tema} · similitud {similitud:.2f}"))
    else:
        sugerencias_tema = sugerir_temas(texto)
        if sugerencias_tema:
            claves = ", ".join(f"«{p}»" for p in dict.fromkeys(p for _, ps in sugerencias_tema for p in ps))
            mensajes.append(("asistente", f"No estoy seguro de haberte entendido. Por las palabras {claves}, ¿tu duda es sobre alguno de estos temas?", None))
            mostrar_opciones([t for t, _ in sugerencias_tema])
        else:
            mensajes.append(("asistente", "No he reconocido ninguna palabra de tu pregunta. Elige el tema de tu duda:", None))
            mostrar_opciones(list(respuestas))
    pintar()

def reiniciar(_=None):
    mensajes.clear()
    opciones.children = []
    mensajes.append(("asistente", "¡Hola! Soy el asistente de PitLineMotor. Pregúntame por tus citas, reparaciones o facturas.", ""))
    pintar()

# Controles
pantalla = widgets.HTML()
opciones = widgets.HBox(layout=widgets.Layout(flex_flow="row wrap", width="560px"))
caja = widgets.Text(placeholder="Escribe tu pregunta y pulsa Enter…",
                    layout=widgets.Layout(width="400px"), continuous_update=False)
boton = widgets.Button(description="Enviar", button_style="danger", icon="paper-plane",
                       layout=widgets.Layout(width="100px"))
boton_reiniciar = widgets.Button(description="Reiniciar", icon="refresh",
                                 layout=widgets.Layout(width="100px"))

def al_enviar(_=None):
    texto = caja.value
    caja.value = ""
    enviar(texto)

caja.observe(lambda cambio: al_enviar() if cambio["new"].strip() else None, names="value")
boton.on_click(al_enviar)
boton_reiniciar.on_click(reiniciar)

# Preguntas rápidas
sugerencias = ["¿Cómo pido cita?", "¿Dónde descargo la factura?",
               "¿Cuánto cuesta la reparación?", "Mi coche espera una pieza"]
botones_rapidos = []
for s in sugerencias:
    b = widgets.Button(description=s, layout=widgets.Layout(width="auto"))
    b.on_click(lambda _, s=s: enviar(s))
    botones_rapidos.append(b)

reiniciar()
display(widgets.VBox([
    pantalla,
    opciones,
    widgets.HBox(botones_rapidos, layout=widgets.Layout(flex_flow="row wrap", width="560px")),
    widgets.HBox([caja, boton, boton_reiniciar]),
]))

Evaluación: 11 de 11 aciertos (100 %)
